# Interactive simulation checks: anemia screening

Verifies the IFA effect on hemoglobin, anemia-status assignment, hemoglobin-screening
coverage (baseline vs the `anemia_screening_vv` scenario), and that the hemoglobin test is
informative. Ported from the research portfolio VnV notebook
`model_18.3_interactive_simulation_anemia_screening`; updated to the current Engine
(`vivarium.engine`) API and to current model behavior.

Note: the source's `ifa_deleted_hemoglobin.exposure` / `first_anc_hemoglobin.exposure`
pipelines were removed, and the raw `hemoglobin_exposure` state column is not populated until
late in the timestep sequence -- so the IFA effect is expressed as IFA-vs-untreated
`hemoglobin.exposure`, and the sim is stepped to `delivery_facility` so the screening/anemia
columns are all populated. The exact test sensitivity/specificity targets (~0.85 / ~0.80) and
the precise hemoglobin measure the test screens are left for researchers to pin.

In [1]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from pathlib import Path

import vivarium_gates_mncnh
from vivarium.engine import InteractiveContext
from vivarium.engine.framework.configuration import build_model_specification

In [2]:
!pip list | grep vivarium

vivarium-artifact                        1.0.7
vivarium-build-utils                     4.4.0
vivarium-cluster-tools                   4.2.12
vivarium-config-tree                     5.0.11
vivarium-dependencies                    1.2.4
vivarium-engine                          5.4.0
vivarium_gates_mncnh                     33.2.dev22+g8e0f39972 /mnt/share/homes/hjafari/repos/vivarium_gates_mncnh
vivarium_gbd_access                      6.0.0
vivarium-gbd-mapping                     6.0.6
vivarium_inputs                          8.0.1
vivarium-public-health                   6.4.2
vivarium-risk-distributions              3.1.7
vivarium-testing-utils                   0.7.6


In [3]:
SPEC_PATH = Path(vivarium_gates_mncnh.__file__).parent / "model_specifications/model_spec.yaml"
COLS = ["anc_attendance", "oral_iron_intervention", "hemoglobin_screening_coverage",
        "ferritin_screening_coverage", "tested_hemoglobin", "anemia_status_during_pregnancy",
        "hemoglobin.exposure"]

def build_sim(scenario=None):
    spec = build_model_specification(SPEC_PATH)
    del spec.configuration.observers
    spec.configuration.population.population_size = 20_000 * 10
    if scenario is not None:
        spec.configuration.intervention.scenario = scenario
    sim = InteractiveContext(spec)
    # Step through all ANC / screening events so the screening + anemia columns are populated.
    ev = sim._builder.time.simulation_event_name()
    while ev() != "delivery_facility":
        sim.step()
    return sim

def anemia_status_from(hb):
    return np.where(hb <= 70, "severe",
           np.where(hb <= 100, "moderate",
           np.where(hb <= 110, "mild", "not_anemic")))

In [4]:
# Baseline scenario
sim = build_sim()
df = sim.get_population(COLS)
df[["anc_attendance", "oral_iron_intervention", "hemoglobin_screening_coverage",
    "tested_hemoglobin", "anemia_status_during_pregnancy"]].head()

2026-08-13 12:26:36.151 | 0:00:12.080887 | INFO     | simulation_1-artifact_manager:_load_artifact:77 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/model41.0/ethiopia.hdf.


2026-08-13 12:26:36.160 | 0:00:12.090153 | INFO     | simulation_1-artifact_manager:_load_artifact:78 - Artifact base filter terms are ['draw == 60'].


2026-08-13 12:26:36.162 | 0:00:12.091941 | INFO     | simulation_1-artifact_manager:_load_artifact:79 - Artifact additional filter terms are None.


2026-08-13 12:26:55.401 | 0:00:31.330300 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:465 - Conflicting information for birth_outcome_probabilities. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-08-13 12:27:07.622 | 0:00:43.551561 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:465 - Conflicting information for lbwsg_paf_on_all_causes.all_cause_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


...

2026-08-13 12:30:48.498 | 0:04:24.427352 | INFO     | simulation_1 - vivarium.engine.framework.engine:step:280 - 2025-01-02 00:00:00


2026-08-13 12:30:57.238 | 0:04:33.167703 | INFO     | simulation_1 - vivarium.engine.framework.engine:step:280 - 2025-01-03 00:00:00


2026-08-13 12:31:11.589 | 0:04:47.518877 | INFO     | simulation_1 - vivarium.engine.framework.engine:step:280 - 2025-01-04 00:00:00


2026-08-13 12:32:05.906 | 0:05:41.835752 | INFO     | simulation_1 - vivarium.engine.framework.engine:step:280 - 2025-01-05 00:00:00


,anc_attendance,oral_iron_intervention,hemoglobin_screening_coverage,tested_hemoglobin,anemia_status_during_pregnancy
0,first_trimester_and_later_pregnancy,ifa,True,adequate,not_anemic
1,first_trimester_only,ifa,False,not_tested,<NA>
2,first_trimester_only,ifa,False,not_tested,<NA>
3,none,no_treatment,False,not_tested,<NA>
4,first_trimester_and_later_pregnancy,ifa,False,not_tested,not_anemic


## IFA raises hemoglobin

In [5]:
# The IFA effect is applied within the `hemoglobin.exposure` pipeline, so IFA-treated
# simulants have a higher hemoglobin exposure than the untreated.
ifa = df.oral_iron_intervention == "ifa"
assert df.loc[ifa, "hemoglobin.exposure"].mean() > df.loc[~ifa, "hemoglobin.exposure"].mean(), \
    "IFA-treated simulants do not have higher hemoglobin than the untreated"

## Anemia status is ordered by hemoglobin

In [6]:
# REVIEWER NOTE (loosened): exact threshold match (against the removed ifa_deleted_hemoglobin
# pipeline) replaced with an ordering-by-hemoglobin check.
# anemia_status_during_pregnancy is assigned to screened simulants from hemoglobin thresholds
# (severe <=70, moderate <=100, mild <=110, else not_anemic). Assert the assigned categories
# are ordered by mean hemoglobin.exposure.
known = ["severe", "moderate", "mild", "not_anemic"]
assigned = df[df.anemia_status_during_pregnancy.isin(known)]
assert len(assigned) > 0, "no simulants have an assigned anemia status"
order = assigned.groupby("anemia_status_during_pregnancy")["hemoglobin.exposure"].mean()
assert order["severe"] < order["moderate"] < order["mild"] < order["not_anemic"], \
    f"anemia-status categories not ordered by hemoglobin: {order.to_dict()}"

## Screening coverage: baseline

In [7]:
# Baseline hemoglobin screening happens at the later-pregnancy ANC visit, so only attendees
# with a later visit are (partially) screened; first-trimester-only and no-ANC are not; and
# ferritin screening is off at baseline.
cov = df.groupby("anc_attendance").hemoglobin_screening_coverage.mean()
assert cov.loc["none"] == 0, "hemoglobin screening occurred among no-ANC simulants at baseline"
later = ["later_pregnancy_only", "first_trimester_and_later_pregnancy"]
assert ((cov.loc[later] > 0) & (cov.loc[later] < 1)).all(), \
    f"expected partial baseline screening among later-visit ANC attendees, got {cov.to_dict()}"
assert cov.get("first_trimester_only", 0) == 0, \
    "first-trimester-only attendees were screened at baseline (screening is at the later visit)"
assert (~df.ferritin_screening_coverage).all(), "ferritin screening should be off at baseline"

## The hemoglobin test is informative

In [8]:
# REVIEWER NOTE (loosened): exact sensitivity/specificity targets (~0.85 / ~0.80, atol 0.05)
# relaxed to > 0.7; the truth basis (hemoglobin.exposure vs first_trimester_hemoglobin_exposure)
# is unconfirmed.
# Among screened simulants, the test should be well better than chance: most truly-low test
# low (sensitivity), most truly-adequate test adequate (specificity). Nominal targets are
# ~0.85 / ~0.80; truth is taken from hemoglobin.exposure (< 100 g/L). Researchers can tighten
# to exact targets and to whatever hemoglobin measure the test actually screens.
tested = df[df.tested_hemoglobin != "not_tested"].copy()
tested["truth"] = np.where(tested["hemoglobin.exposure"] < 100, "low", "adequate")
sens = (tested.loc[tested.truth == "low", "tested_hemoglobin"] == "low").mean()
spec = (tested.loc[tested.truth == "adequate", "tested_hemoglobin"] == "adequate").mean()
assert sens > 0.7, f"hemoglobin-test sensitivity {sens:.3f} unexpectedly low (target ~0.85)"
assert spec > 0.7, f"hemoglobin-test specificity {spec:.3f} unexpectedly low (target ~0.80)"

## Screening coverage: `anemia_screening_vv` scale-up scenario

In [9]:
# In the anemia-screening VnV scenario, later-pregnancy ANC attendees are all screened for
# hemoglobin and no-ANC simulants are still never screened.
vv = build_sim(scenario="anemia_screening_vv")
vv_cov = vv.get_population(["anc_attendance", "hemoglobin_screening_coverage"]) \
    .groupby("anc_attendance").hemoglobin_screening_coverage.mean()
assert vv_cov.loc["none"] == 0, "hemoglobin screening among no-ANC simulants in anemia_screening_vv"
later = ["later_pregnancy_only", "first_trimester_and_later_pregnancy"]
assert (vv_cov.loc[later] == 1).all(), \
    f"expected 100% hemoglobin screening at later-visit ANC in anemia_screening_vv, got {vv_cov.to_dict()}"

2026-08-13 12:33:54.063 | 0:07:29.992727 | INFO     | simulation_2-artifact_manager:_load_artifact:77 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/model41.0/ethiopia.hdf.


2026-08-13 12:33:54.066 | 0:07:29.995298 | INFO     | simulation_2-artifact_manager:_load_artifact:78 - Artifact base filter terms are ['draw == 60'].


2026-08-13 12:33:54.067 | 0:07:29.996742 | INFO     | simulation_2-artifact_manager:_load_artifact:79 - Artifact additional filter terms are None.


2026-08-13 12:34:13.533 | 0:07:49.463160 | WARNING  | simulation_2-values_manager:_warn_if_overriding_resources:465 - Conflicting information for birth_outcome_probabilities. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-08-13 12:34:24.423 | 0:08:00.352490 | WARNING  | simulation_2-values_manager:_warn_if_overriding_resources:465 - Conflicting information for lbwsg_paf_on_all_causes.all_cause_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


...

2026-08-13 12:36:13.914 | 0:09:49.843674 | INFO     | simulation_2 - vivarium.engine.framework.engine:step:280 - 2025-01-01 00:00:00


2026-08-13 12:37:03.031 | 0:10:38.961050 | INFO     | simulation_2 - vivarium.engine.framework.engine:step:280 - 2025-01-02 00:00:00


2026-08-13 12:37:08.322 | 0:10:44.251472 | INFO     | simulation_2 - vivarium.engine.framework.engine:step:280 - 2025-01-03 00:00:00


2026-08-13 12:37:17.137 | 0:10:53.066729 | INFO     | simulation_2 - vivarium.engine.framework.engine:step:280 - 2025-01-04 00:00:00


2026-08-13 12:37:51.315 | 0:11:27.244239 | INFO     | simulation_2 - vivarium.engine.framework.engine:step:280 - 2025-01-05 00:00:00
